####**Business Objective**

#####The purpose of this notebook is to transform the machine learning outputs into a business intelligence dataset that supports interactive dashboards in Power BI. The exported data will allow marketing, operations, customer service, and senior management teams to monitor customer sentiment, identify recurring issues, and evaluate business performance in real time.

####**Technical Objective**

#####This notebook will:

-Prepare a clean Power BI dataset\
-Create Date Dimension\
-Create KPI Table\
-Create Summary Tables\
-Create Country Summary\
-Create Category Summary\
-Create Monthly Summary\
-Create Driver Summary\
-Save all tables separately\
-Save final dashboard dataset

####Import Libraries

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import re
import numpy as np
import pandas as pd

####Load Dataset

In [2]:
df = pd.read_csv(
    "/content/final_sentiment_dataset.csv"
)

print(df.shape)

df.head()

(18348, 26)


,review_id,product_category,timestamp,country,rating,review,sentiment,date,year,month,...,word_count,character_count,sentence_count,avg_word_length,sentiment_driver,rating_group,review_density,weekend,long_review,target
0,REV-50BCBCD9,Sports,2024-09-16 13:44:26+00:00,US,1,"I registered on the website, tried to order a ...",Negative,2024-09-16,2024,9,...,52.0,364.0,6.0,7.000000,Website,Low,8.666667,0.0,1.0,0.0
1,REV-6D2B2651,Toys,2024-09-16 18:26:46+00:00,GB,1,Had multiple orders one turned up and driver h...,Negative,2024-09-16,2024,9,...,31.0,205.0,2.0,6.612903,Delivery,Low,15.500000,0.0,0.0,0.0
2,REV-F7E80372,Toys,2024-09-16 21:47:39+00:00,GB,1,I informed these reprobates that I WOULD NOT B...,Negative,2024-09-16,2024,9,...,53.0,351.0,4.0,6.622642,Delivery,Low,13.250000,0.0,1.0,0.0
3,REV-ED2B173F,Sports,2024-09-17 07:15:49+00:00,AU,1,I have bought from Amazon before and no proble...,Negative,2024-09-17,2024,9,...,40.0,278.0,4.0,6.950000,Product Quality,Low,10.000000,0.0,0.0,0.0
4,REV-E48A7AB9,Fashion,2024-09-16 18:37:17+00:00,GB,1,If I could give a lower rate I would! I cancel...,Negative,2024-09-16,2024,9,...,48.0,337.0,7.0,7.020833,Price,Low,6.857143,0.0,0.0,0.0


####Clean Dataset

In [3]:
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce",
    utc=True
)

df["timestamp"] = df["timestamp"].dt.tz_localize(None)

df = df.dropna(subset=["timestamp"])

#####Create Date

In [4]:
df["date"] = df["timestamp"].dt.date

df["year"] = df["timestamp"].dt.year

df["month"] = df["timestamp"].dt.month

df["month_name"] = df["timestamp"].dt.month_name()

df["quarter"] = df["timestamp"].dt.quarter

df["weekday"] = df["timestamp"].dt.day_name()

df["hour"] = df["timestamp"].dt.hour

####PowerBI Dataset

#####Select only required columns

In [6]:
powerbi_df = df[
[
"review_id",
"date",
"year",
"quarter",
"month",
"month_name",
"weekday",
"hour",
"country",
"language",
"product_category",
"rating",
"sentiment",
"sentiment_driver",
"review_length",
"word_count",
"review"
]
]

In [8]:
import os
os.makedirs('data/powerbi', exist_ok=True)
powerbi_df.to_csv(
"data/powerbi/shopease_powerbi_clean.csv",
index=False
)

####Create Date Dimension

In [9]:
date_dimension = pd.DataFrame({

"Date":

pd.date_range(

powerbi_df["date"].min(),

powerbi_df["date"].max()

)

})

#####Create

In [ ]:
date_dimension["Year"] = date_dimension["Date"].dt.year

date_dimension["Quarter"] = date_dimension["Date"].dt.quarter

date_dimension["Month"] = date_dimension["Date"].dt.month

date_dimension["Month Name"] = date_dimension["Date"].dt.month_name()

date_dimension["Weekday"] = date_dimension["Date"].dt.day_name()

date_dimension["Week Number"] = date_dimension["Date"].dt.isocalendar().week

In [10]:
date_dimension.to_csv(

"data/powerbi/date_dimension.csv",

index=False

)

####Create KPI Table

In [11]:
kpi = pd.DataFrame({

"KPI":[

"Total Reviews",

"Average Rating",

"Negative Reviews",

"Positive Reviews",

"Neutral Reviews",

"Countries",

"Categories"

],

"Value":[

len(df),

round(df["rating"].mean(),2),

(df["sentiment"]=="Negative").sum(),

(df["sentiment"]=="Positive").sum(),

(df["sentiment"]=="Neutral").sum(),

df["country"].nunique(),

df["product_category"].nunique()

]

})

In [12]:
kpi.to_csv(

"data/powerbi/kpi_table.csv",

index=False

)

####Country Summary

In [13]:
country_summary = (

df

.groupby(

"country"

)

.agg(

Total_Reviews=("review_id","count"),

Average_Rating=("rating","mean"),

Negative=("sentiment",

lambda x:(x=="Negative").sum()),

Positive=("sentiment",

lambda x:(x=="Positive").sum()),

Neutral=("sentiment",

lambda x:(x=="Neutral").sum())

)

.reset_index()

)

In [14]:
country_summary.to_csv(

"data/powerbi/country_summary.csv",

index=False

)

#####**Business Insight**

#####This table enables management to compare customer satisfaction across countries. It can highlight regions with higher complaint volumes, lower average ratings, or declining customer experience, supporting targeted operational improvements.

####Product Category Summary

In [15]:
category_summary = (

df

.groupby(

"product_category"

)

.agg(

Total_Reviews=("review_id","count"),

Average_Rating=("rating","mean"),

Negative=("sentiment",

lambda x:(x=="Negative").sum()),

Positive=("sentiment",

lambda x:(x=="Positive").sum()),

Neutral=("sentiment",

lambda x:(x=="Neutral").sum())

)

.reset_index()

)

In [16]:
category_summary.to_csv(

"data/powerbi/category_summary.csv",

index=False

)

#####**Business Insight**

#####This summary identifies which product categories receive the highest volume of complaints and which maintain stronger customer satisfaction. These insights can guide product quality initiatives and category-level performance reviews.

####Driver Summary

In [17]:
driver_summary = (

df

.groupby(

"sentiment_driver"

)

.size()

.reset_index(name="Reviews")

.sort_values(

"Reviews",

ascending=False

)

)

In [18]:
driver_summary.to_csv(

"data/powerbi/driver_summary.csv",

index=False

)

#####Business Insight

#####This table highlights the primary drivers of customer feedback, such as delivery, product quality, pricing, or customer support. Monitoring these drivers helps prioritise improvement initiatives based on customer concerns.

#####Monthly Summary

In [19]:
monthly = (

df

.groupby(

["year","month_name","sentiment"]

)

.size()

.reset_index(name="Reviews")

)

In [20]:
monthly.to_csv(

"data/powerbi/monthly_summary.csv",

index=False

)

#####**Business Insight**

#####Tracking sentiment over time allows ShopEase Europe to evaluate the impact of operational changes, seasonal trends, marketing campaigns, or product launches on customer satisfaction.

####Language Summary

In [21]:
language = (

df["language"]

.value_counts()

.reset_index()

)

language.columns = [

"Language",

"Reviews"

]

In [22]:
language.to_csv(

"data/powerbi/language_summary.csv",

index=False

)